# Nutrition5K Data Exploration
This notebook is for exploring the Nutrition5K dataset and understanding what we have to work with.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os, sys
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch

# Check for GPU
if torch.cuda.is_available():
    print("✓ GPU available:", torch.cuda.get_device_name(0))
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print("✓ Apple Silicon GPU (MPS) available")
else:
    print("✗ No GPU found — running on CPU")


In [ ]:
# ── Set data paths ────────────────────────────────────────────────────────────
KAGGLE_ROOT = '/kaggle/input'
LOCAL_DATA  = 'data/archive'

if os.path.exists(KAGGLE_ROOT):
    # On Kaggle — find the mounted dataset folder automatically
    candidates = [d for d in os.listdir(KAGGLE_ROOT) if 'nutrition' in d.lower()]
    if candidates:
        data_dir = os.path.join(KAGGLE_ROOT, candidates[0])
        print(f"✓ Kaggle: using dataset folder '{candidates[0]}'")
    else:
        # Fallback: list all available inputs so you can pick the right one
        available = os.listdir(KAGGLE_ROOT)
        raise FileNotFoundError(
            f"No nutrition5k dataset found under {KAGGLE_ROOT}.\n"
            f"Available inputs: {available}\n"
            "Add the dataset via 'Add Input' in the Kaggle sidebar."
        )
else:
    data_dir = LOCAL_DATA
    print("✓ Local environment — using:", data_dir)

img_dir              = os.path.join(data_dir, 'imagery/realsense_overhead')
ingredients_csv      = os.path.join(data_dir, 'dish_ingredients.csv')
nutrition_csv        = os.path.join(data_dir, 'dish_nutrition_values.csv')
ingredients_meta_csv = os.path.join(data_dir, 'ingredients_metadata.csv')
print("✓ Data paths set:", data_dir)


In [ ]:
# ── Verify paths exist ────────────────────────────────────────────────────────
for label, path in [
    ("Image dir", img_dir),
    ("dish_ingredients.csv", ingredients_csv),
    ("dish_nutrition_values.csv", nutrition_csv),
    ("ingredients_metadata.csv", ingredients_meta_csv),
]:
    status = "✓" if os.path.exists(path) else "✗ MISSING"
    print(f"{status}  {label}: {path}")


In [ ]:
# Checking out the CSV files
df_ingredients = pd.read_csv(ingredients_csv)
df_nutrition = pd.read_csv(nutrition_csv)
df_ingredients_meta = pd.read_csv(ingredients_meta_csv)
print('dish_ingredients.csv:')
display(df_ingredients.head())
print('dish_nutrition_values.csv:')
display(df_nutrition.head())
print('ingredients_metadata.csv:')
display(df_ingredients_meta.head())

In [ ]:
# Let's see how many dishes and images we have
dish_folders = [f for f in os.listdir(img_dir) if f.startswith('dish_')]
print(f'Total number of dish folders: {len(dish_folders)}')
# Show a few sample images
sample_dish = dish_folders[0] if dish_folders else None
if sample_dish:
    sample_path = os.path.join(img_dir, sample_dish)
    sample_imgs = [f for f in os.listdir(sample_path) if f.endswith('.jpg') or f.endswith('.png')]
    print(f'Sample images in {sample_dish}:', sample_imgs[:3])
    for img_name in sample_imgs[:3]:
        img = Image.open(os.path.join(sample_path, img_name))
        plt.imshow(img)
        plt.title(f'{sample_dish} - {img_name}')
        plt.axis('off')
        plt.show()

In [ ]:
# Checking the distribution of calories in the dataset
plt.figure(figsize=(8,4))
plt.hist(df_nutrition['calories'], bins=30, color='skyblue', edgecolor='black')
plt.xlabel('Calories')
plt.ylabel('Number of Dishes')
plt.title('Calories Distribution in Nutrition5K')
plt.show()